# OCR(Optical Character Recognition)
    -  이미지 속의 문자들을 컴퓨터가 인식할 수 있도록 디지털 문자로 변환해주는 기술입니다.
    문서를 디지털화 하는 가장 기본적인 기술로 자리잡았으며, 이미지, PDF, 문서속 문자를 인식하고
    편집 및 검색이 가능하게 하는 AI기반의 기술을 의미합니다.

In [3]:
from dotenv import load_dotenv
import os

from langchain_openai.chat_models.base import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-VZytJ5BN3ErM


## 1. llm 준비

In [4]:
ocr_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [7]:
# base64: 컴퓨터에서 8비트 바이너리(이진) 데이터인 이미지, 실행파일 등을 인쇄 가능한 
# 64개의 아스키코드(A-Z, a-z, 0-9, +, /)로만 변환하여 안전하게 전송 또는 저장하는 인코딩 방식
import base64

def encode_image(image_path):
    with open(image_path, "rb") as image:
        return base64.b64encode(image.read()).decode("utf-8")

In [13]:
base_path = r"C:\back_0900_ksh\python\resource"
receipt_name = r"receipt_01.png"
receipt_path = os.path.join(base_path, receipt_name)

receipt_path

'C:\\back_0900_ksh\\python\\resource\\receipt_01.png'

In [14]:
base64_receipt = encode_image(receipt_path)

## 프롬프트 준비

In [38]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an advanced vision model.

        Your goal is to extract ALL text from the image.

        IMPORTANT:
        - Do NOT miss any text
        - If text is unclear, infer the most likely correct text
        - Preserve all lines, even unusual ones
        - Do NOT drop promotional or "증정품" lines
        """),
    ("human", [
                {
                    "type": "text",
                    "text": """
        Extract ALL visible text from this receipt.

        Rules:
        - Include EVERY line (even if it looks unimportant)
        - Keep line breaks
        - Do NOT omit anything
        - If a word is unclear, guess the most likely correct word

        Output ONLY the extracted text.
        """
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "data:image/jpeg;base64,{encoded_image}"
                    }
                }
            ])
        ])

In [39]:
ocr_chain = prompt | ocr_llm

In [40]:
result = ocr_chain.invoke({
    "encoded_image": base64_receipt
})

In [42]:
print(result.content)

2026/03/01 정*비 NO:14002

*정부방침에 의해 교환/환불은
반드시 영수증을 지참하셔야 하며,
카드결제는 30일(03월31일) 이내
카드와 영수증 지참 시 가능합니다
선도민감상품 및 일부상품 제외
----------------------------------------
에그소시지샌드 1      2,800
참치김치김밥 1      3,500
정성가득비빔밥 1      4,600
합계수량/금액 3     10,900
KT 멤버십 할인     -1,000
----------------------------------------
과세 매출      9,000
부가세         900
합    계      9,900
GS QR/바코드 결제할인  -3,000
모바일상품권      3,400
현    금      3,500


## JSON 형태로 변환

In [46]:
from typing import List 
from pydantic import BaseModel, Field

In [47]:
class ReceiptItem(BaseModel):
    name: str = Field(description="The exact product name from the text. Copy verbatim.")
    count: int = Field(description="The quantity.")
    price: int = Field(description="The price text exactily as printed (e.g., '1,600' or '증정품').")

class ReceiptInfo(BaseModel):
    store_name: str = Field(description="Store name")
    date: str = Field(description="YYYY-MM-DD")
    items: List[ReceiptItem] = Field(description="List of purchansed items.")
    amount: int = Field(description="Discounted price")
    total_amount: int = Field(description="Price after discount")

In [48]:
parse_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 리턴값은 무조건 ReceiptInfo 타입
structured_llm = parse_llm.with_structured_output(ReceiptInfo)

### ※문제점

In [49]:
json_ocr_chain = prompt | structured_llm

In [50]:
json_ocr_result =  json_ocr_chain.invoke({
    "encoded_image": base64_receipt
})

C:\back_0900_ksh\python\workspace\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReceiptInfo(store_name='...9900, total_amount=9900), input_type=ReceiptInfo])
  return self.__pydantic_serializer__.to_python(


In [56]:
# .model_dump(): 데이터 역직렬화
json_ocr_result.model_dump()

{'store_name': '정*비',
 'date': '2026-03-01',
 'items': [{'name': '에고소시지샌드', 'count': 1, 'price': 2800},
  {'name': '참치김치김밥', 'count': 1, 'price': 3500},
  {'name': '정성가득비빔밥', 'count': 1, 'price': 4600},
  {'name': '합계수량/금액', 'count': 3, 'price': 10900},
  {'name': 'KT 멤버십 할인', 'count': 1, 'price': -1000},
  {'name': '과세 매출', 'count': 1, 'price': 9000},
  {'name': '부가세', 'count': 1, 'price': 900},
  {'name': '합 계', 'count': 1, 'price': 9900},
  {'name': 'GS QR/바코드 결제할인', 'count': 1, 'price': -3000},
  {'name': '모바일상품권', 'count': 1, 'price': 3400},
  {'name': '현 금', 'count': 1, 'price': 3500}],
 'amount': 9900,
 'total_amount': 9900}

### 문제 해결

### 1. 프롬프트 생성

In [57]:
parsing_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a deterministic parser.

        STRICT RULES:
        - Each line represents EXACTLY one item.
        - NEVER merge lines.
        - NEVER split lines.
        - NEVER infer or guess missing values.
        - Copy text exactly as written.

        You must process input line-by-line.
        """
            ),
            (
                "human",
                """
        Here is the raw receipt text:

        {receipt_text}

        Instructions:
        1. Only extract lines that match this pattern:
           [product name] [count] [price or '증정품']

        2. Ignore all other lines.

        3. Output must be JSON array:
        [
          {{"name": "...", "count": ..., "price": "..."}}
        ]

        4. Do not change numbers.
        5. Do not duplicate items.
        6. Do not create new items.

        Output JSON only.
        """
    )
])

### 2. 체인 생성

In [64]:
def transform_output(input_messages):
    return {"receipt_text": input_messages.content}

def transform_output_model_dump(input_messages):
    return input_messages.model_dump()

In [59]:
parse_chain = parsing_prompt | structured_llm

In [67]:
final_chian = ocr_chain | RunnableLambda(transform_output) | parse_chain | RunnableLambda(transform_output_model_dump)

In [68]:
final_result = final_chian.invoke({
    "encoded_image": base64_receipt
})

C:\back_0900_ksh\python\workspace\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ReceiptInfo(store_name='...1000, total_amount=9900), input_type=ReceiptInfo])
  return self.__pydantic_serializer__.to_python(


In [72]:
final_result

{'store_name': '정*비',
 'date': '2026-03-01',
 'items': [{'name': '에그소시지샌드', 'count': 1, 'price': 2800},
  {'name': '참치김치김밥', 'count': 1, 'price': 3500},
  {'name': '정성가득비빔밥', 'count': 1, 'price': 4600}],
 'amount': -1000,
 'total_amount': 9900}

## 객체 탐지

In [75]:
base_path = r"C:\back_0900_ksh\python\resource"

penguin_name = r"penguin.jpg"
cat_name = r"cat.png"

penguin_path = os.path.join(base_path, penguin_name)
cat_path = os.path.join(base_path, cat_name)

base64_penguin = encode_image(penguin_path)
base64_cat = encode_image(cat_path)

In [97]:
classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an object detector.

        You must output exactly:
        - List all visible objects in the image.
        - ["cat"]

        Rules:
        - Output must as be JSON array
        - Use simple nouns
        - No duplicates
    """),
    ("human", [
        {
            "type": "text",
            "text": """
                List all objects in the image.

                Return format example:
                ["cat", "dog"]

                Do not add anything else.
            """
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/jpeg;base64,{encoded_image}"
            }
        }
    ])
])

In [105]:
ocr_llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [106]:
import json

def json_parser(input_messages):
    return json.loads(input_messages.content)

In [108]:
classifier_chain = classifier_prompt | ocr_llm | RunnableLambda(json_parser)

In [109]:
cat_result = classifier_chain.invoke({
    "encoded_image": base64_cat
})

In [111]:
cat_result

['cat']

In [112]:
penguin_result = classifier_chain.invoke({
    "encoded_image": base64_penguin
})

In [113]:
penguin_result

['penguin']

## 상황 브리핑

In [114]:
brief_llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

In [161]:
brief_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are an expert visual scene analyst specification.

        Your task:
        - Analyze the provided image deeply and describe the scene in detail.

        Rules:
        - **Focus on:** Objects, actions, relationships, and precise colors (especially hair color, clothing color, accessories, and background colors).
        - **Detailed Breakdown:** Do not generalize. Identify specific entities and break down their appearance, posture, and environmental details.
        - **Length Rule:** Please provide a detailed explanation. The response must be at least 10 lines long. Ensure each logical point is separated by a line break to maintain readability.
        - **Constraint:** Do NOT guess unknown facts or speculate. Describe only what is visible.
        - **Language:** You must answer in Korean.
    """),
    ("human", [
        {
            "type": "text",
            "text": "Describe what is happening in this image in full detail, focusing on appearance, actions, colors, and relationships."
        },
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/jpeg;base64,{encoded_image}"
            }
        }
    ])
])

In [157]:
brief_chain = brief_prompt | brief_llm

In [158]:
yujung_name = "brief_yujung.png"
yujung_path = os.path.join(base_path, yujung_name)
base64_yujung = encode_image(yujung_path)

In [159]:
brief_result = brief_chain.invoke({
    "encoded_image": base64_yujung
})

In [160]:
print(brief_result.content)

이 이미지에서는 한 남성이 파란색 후드티를 입고 있으며, 두 팔을 높이 들고 있는 모습이 보입니다. 그의 표정은 매우 기쁜 듯하며, 무언가를 축하하는 분위기입니다.

그는 원형 테이블에 앉아 있으며, 테이블 위에는 초가 꽂힌 초콜릿 케이크가 놓여 있습니다. 케이크 옆에는 작은 상자와 두 개의 컵이 있으며, 그 옆에는 음료 캔과 병이 보입니다. 이로 보아, 생일이나 축하하는 자리가 마련된 것으로 보입니다.

주변에는 여러 사람이 함께 앉아 있으며, 그들은 박수를 치거나 환호하는 듯한 모습입니다. 참석자들은 대체로 웃고 있으며, 전체적인 분위기는 화기애애하고 즐거운 분위기입니다.

테이블은 연두색 테이블보로 덮여 있으며, 그 위에는 여러 개의 물건이 놓여 있습니다. 배경에는 여러 사람과 함께 식당이나 연회장과 같은 공간이 보입니다.

사람들은 대부분 캐주얼한 옷차림을 하고 있으며, 공간은 실내로 밝고 깔끔하게 정리되어 있습니다. 벽에는 큰 액자가 걸려 있거나 장식이 되어 있는 것 같지만, 정확한 내용은 알 수 없습니다.

전체적으로, 이 장면은 사람들이 모여 기념일이나 특별한 날을 축하하는 순간을 포착한 것으로, 참여자들은 모두 활기차고 즐거운 시간을 보내고 있는 것 같습니다.


### QA

In [162]:
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a question-answering assistant.

        Answer based ONLY on the given description.
        Do NOT assume anything beyond the description.
    """),
    ("human", """
        Description:
        {description}

        Question:
        {question}
    """)
    ])

In [163]:
qa_chain = qa_prompt | brief_llm

In [164]:
qa_result = qa_chain.invoke({
    "description": brief_result.content,
    "question": "남자가 입은 후드티의 색깔은?"
})

In [166]:
qa_result

AIMessage(content='파란색입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 400, 'total_tokens': 405, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_e7fdabdb53', 'id': 'chatcmpl-DgmqibakpHfEMAQXOJcrlKtqx0Odn', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3a05-e0f7-73c3-b9a3-64390f5f1678-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 400, 'output_tokens': 5, 'total_tokens': 405, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [167]:
def extract_question(input_value):
    return input_value["question"]

def format_qa_input(input_value):
    return {
        "description": input_value["description"].content,
        "question": input_value["question"]
    }

In [168]:
final_qa_chain = (
    {
        "description": brief_chain,
        "question": RunnableLambda(extract_question)
    } 
    | RunnableLambda(format_qa_input) 
    | qa_chain
)

In [169]:
final_qa_result = final_qa_chain.invoke({
    "encoded_image": base64_yujung,
    "question": "남자가 입고 있는 후드티의 색깔은?"
})

In [173]:
final_qa_result.content

'파란색입니다.'

In [174]:
final_qa_result = final_qa_chain.invoke({
    "encoded_image": base64_yujung,
    "question": "케이크의 촛불 개수는?"
})

In [175]:
final_qa_result.content

'설명에는 케이크에 작은 초가 꽂혀 있다고만 언급되어 있으며, 촛불의 개수에 대한 정보는 제공되지 않았습니다.'